In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import keras
import torch
import numpy as np
import albumentations as A

from pathlib import Path
from typing import Sequence, Tuple, List, Optional

from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from agx_core.transforms import (
    LogTransform,
    GammaCorrection,
    Deskew,
)

keras.utils.set_random_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


product_sets = [
    ("midmeki", "top", "Bread"),
    ("meki", "top", "ChocotrioNestleOriginal"),
    ("midmeki", "top", "LaTuaPastaGlassJars"),
    ("meki", "top", "MEKIONETunaCan"),
    ("sidemeki", "left", "SIDEMEKIBeerLeft"),
    ("sidemeki", "right", "SIDEMEKIBeerRight"),
    ("sidemeki", "left", "TamarindPasteLeft"),
    ("sidemeki", "right", "TamarindPasteRight"),
    ("meki", "top", "TunaFlakes"),
    ("midmeki", "top", "VineLeavesCan"),
]

# Build vocabularies: sorted for deterministic ID assignment
MACHINES = sorted(set(e[0] for e in product_sets))  # ['meki', 'midmeki', 'sidemeki']
VIEWS = sorted(set(e[1] for e in product_sets))  # ['left', 'right', 'top']
PRODUCTS = sorted(set(e[2] for e in product_sets))  # ['Bread', 'Choco...', ...]

MACHINE_TO_ID = {m: i for i, m in enumerate(MACHINES)}
VIEW_TO_ID = {v: i for i, v in enumerate(VIEWS)}
PRODUCT_TO_ID = {p: i for i, p in enumerate(PRODUCTS)}


class StratifiedProductSampler(torch.utils.data.Sampler):
    """Yields batches where all samples are from the same product.

    Cycles through products, yielding full batches from each.
    Ensures manifold interpolation (roll) pairs same-product samples.
    """

    def __init__(self, dataset: ConcatDataset, batch_size: int, shuffle: bool = True):
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Map each sample index to its sub-dataset
        self.product_indices = []  # List[List[int]] — indices per product
        offset = 0
        for sub_ds in dataset.datasets:
            indices = list(range(offset, offset + len(sub_ds)))
            self.product_indices.append(indices)
            offset += len(sub_ds)

    def __iter__(self):
        # Shuffle within each product
        all_batches = []
        for indices in self.product_indices:
            if self.shuffle:
                perm = torch.randperm(len(indices)).tolist()
                indices = [indices[i] for i in perm]

            # Chunk into batches
            for i in range(0, len(indices) - self.batch_size + 1, self.batch_size):
                all_batches.append(indices[i : i + self.batch_size])

        # Shuffle batch ORDER (but each batch is still single-product)
        if self.shuffle:
            perm = torch.randperm(len(all_batches)).tolist()
            all_batches = [all_batches[i] for i in perm]

        for batch in all_batches:
            yield batch

    def __len__(self):
        total = 0
        for indices in self.product_indices:
            total += len(indices) // self.batch_size
        return total


# Usage:
# sampler = StratifiedProductSampler(ds_train, batch_size=16, shuffle=True)
# loader_train = DataLoader(ds_train, batch_sampler=sampler)


class XRayProductDataset(Dataset):
    """Multi-product X-ray dataset with categorical condition encoding.

    Loads images from: {root_dir}/{product_name}/{suffix_path}/
    Returns condition vector: [machine_id, view_id, product_id]

    Args:
        root_dir: Base data directory containing product folders.
        product_set: Tuple of (machine_name, view_name, product_name).
        suffix_path: Subdirectory path within product folder (e.g., "Clean/train").
        transform: Albumentations transform pipeline.
        extensions: Image file extensions to glob for.
    """

    def __init__(
        self,
        root_dir: Path,
        product_set: Tuple[str, str, str],
        suffix_path: str = "Clean/train",
        transform=None,
        extensions: Sequence[str] = ("*.bmp", "*.png", "*.jpg"),
        device: Optional[torch.device] = None,  # Uses cuda if available
    ):
        self.root_dir = Path(root_dir)
        self.transform = transform

        machine, view, product = product_set
        self.machine_id = MACHINE_TO_ID[machine]
        self.view_id = VIEW_TO_ID[view]
        self.product_id = PRODUCT_TO_ID[product]
        self._device = (
            device
            if device
            else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        )

        # Condition vector: [machine_id, view_id, product_id]
        self._condition = np.array(
            [self.machine_id, self.view_id, self.product_id],
            dtype=np.float32,
        )

        # Discover image files
        image_dir = self.root_dir / product / suffix_path
        self.image_files: List[Path] = []
        for ext in extensions:
            self.image_files.extend(image_dir.glob(ext))
        self.image_files.sort()  # deterministic order

        if len(self.image_files) == 0:
            raise FileNotFoundError(
                f"No images found in {image_dir} with extensions {extensions}"
            )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image = Image.open(self.image_files[idx]).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))["image"][..., np.newaxis]

        image = torch.tensor(image, dtype=torch.float32, device=self._device)
        condition = torch.tensor(
            self._condition, dtype=torch.float32, device=self._device
        )

        return (image, condition), image

    @property
    def condition_vector(self) -> np.ndarray:
        """The fixed condition vector for this dataset."""
        return self._condition.copy()

    def __repr__(self):
        return (
            f"XRayProductDataset("
            f"product={PRODUCTS[self.product_id]}, "
            f"machine={MACHINES[self.machine_id]}, "
            f"view={VIEWS[self.view_id]}, "
            f"n_images={len(self)})"
        )


def build_multi_product_dataset(
    root_dir: Path,
    product_sets: List[Tuple[str, str, str]],
    suffix_path: str = "Clean/train",
    transform=None,
) -> ConcatDataset:
    """Build a ConcatDataset from multiple product sets.

    Args:
        root_dir: Base data directory.
        product_sets: List of (machine, view, product) tuples.
        suffix_path: Subdirectory within each product folder.
        transform: Shared transform pipeline.

    Returns:
        ConcatDataset containing all products. Use with standard DataLoader
        for mixed batches, or with a stratified sampler for same-product batches.
    """
    datasets = []
    for product_set in product_sets:
        try:
            ds = XRayProductDataset(
                root_dir=root_dir,
                product_set=product_set,
                suffix_path=suffix_path,
                transform=transform,
            )
            datasets.append(ds)
            print(f"  ✓ {ds}")
        except FileNotFoundError as e:
            print(f"  ✗ Skipping {product_set[2]}: {e}")

    return ConcatDataset(datasets)

In [ ]:
def train_transforms(img_size, mean=[0.5], std=[0.5]):
    return A.Compose(
        [
            # Geometry correction (sometimes raw)
            Deskew(p=0.85),
            # Position/size variation
            A.OneOf(
                [
                    A.Pad((25, 25), 255),
                    A.Pad((50, 50), 255),
                    A.Pad((75, 75), 255),
                    A.NoOp(),
                ],
                p=0.75,
            ),
            # Physics preprocessing (always on, params vary)
            LogTransform(epsilon=1, invert=True, p=1.0),
            A.OneOf(
                [
                    GammaCorrection(center=0.8, gain=5),
                    GammaCorrection(center=0.85, gain=7),
                    GammaCorrection(center=0.9, gain=9),
                    GammaCorrection(center=0.95, gain=11),
                ],
                p=1.0,
            ),
            A.Resize(img_size, img_size),
            # Geometric augmentation
            A.Affine(scale=(0.88, 0.97), rotate=(-90, 90), shear=(-5, 5), p=0.5),
            A.RandomRotate90(p=0.5),
            # Sensor/acquisition noise
            A.OneOf(
                [
                    A.GaussianBlur(blur_range=(1, 3)),
                    A.MotionBlur(blur_range=(3, 7)),
                    A.GaussNoise(std_range=(0.01, 0.03)),
                    A.CoarseDropout(
                        num_holes_range=(1, 3),
                        hole_height_range=(1, 5),
                        hole_width_range=(1, img_size),
                        fill=255,  # Image is now inverted, dead pixels are black, inverted are white
                    ),
                    A.ShotNoise(scale_range=(0.01, 0.5)),
                ],
                p=0.3,
            ),
            # Subtle intensity drift (calibration variation)
            A.RandomBrightnessContrast(
                brightness_range=(-0.05, 0.05), contrast_range=(-0.05, 0.05), p=0.2
            ),
            A.Normalize(mean=mean, std=std),
        ]
    )


def valid_transforms(img_size, mean=[0.5], std=[0.5]):
    return A.Compose(
        [
            # Validation: deterministic, canonical preprocessing
            Deskew(p=1.0),
            LogTransform(epsilon=1, invert=True, p=1.0),
            GammaCorrection(center=0.9, gain=9, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=mean, std=std),
        ]
    )

In [ ]:
img_size = 224
res = img_size // 2**5

img_shape = (None, img_size, img_size, 1)
cond_shape = (None, 3)

root_dir = Path("../data/products")
img_size = 224

# Build datasets
ds_train = build_multi_product_dataset(
    root_dir, product_sets, suffix_path="Clean/train",
    transform=train_transforms(img_size),
)
ds_valid = build_multi_product_dataset(
    root_dir, product_sets, suffix_path="Clean/val",
    transform=valid_transforms(img_size),
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    X, y = ds_train[idx+600]
    image = X[0].cpu().numpy()
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from keras.optimizers import Adam

from agx_core.models.reversed_autoencoder import (
    MobileNetV3SmallEncoder,
    MobileNetV3SmallDecoder,
)
from agx_core.models.reversed_autoencoder.layers import CompositeConditionEncoder
from agx_torch.models.reversed_autoencoder.model import ReversedAutoencoder

spatial_temp = 3.0
enc = MobileNetV3SmallEncoder(latent_size=512, weights="imagenet")
dec = MobileNetV3SmallDecoder(target_shape=img_shape[1:])
cond = CompositeConditionEncoder([3, 3, len(PRODUCTS)])
ra = ReversedAutoencoder(
    enc, dec, condition=cond, beta_kld=0.7, spatial_temp=spatial_temp
)

ra.build([img_shape, cond_shape])
ra.compile(Adam(learning_rate=1e-6), Adam(learning_rate=1e-3))

ra.summary()

loader_train = DataLoader(ds_train, batch_size=32, shuffle=True)
loader_valid = DataLoader(ds_valid, batch_size=32)

In [ ]:
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any

rec_dir = Path("reconstructions")
rec_dir.mkdir(exist_ok=True)

plot_every = 5
rec_test_samples = 10

iter_train = iter(loader_train)
(samples, conds), _ = next(iter_train)
n_samples = samples.shape[0]

while n_samples < rec_test_samples:
    (_s, _c), _ = next(iter_train)
    samples = keras.ops.concatenate([samples, _s])
    conds = keras.ops.concatenate([conds, _c])
    n_samples = samples.shape[0]

if n_samples > rec_test_samples:
    samples = samples[:rec_test_samples]
    conds = conds[:rec_test_samples]

import numpy as np

x = np.linspace(-3, 3, 100)
standard_normal = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2)


def plot_on_step_end(step: int, logs: Optional[Dict[str, Any]] = None):
    # plot reconstruction every n epochs/batches
    if (step + 1) % plot_every != 0:
        return

    with torch.no_grad():
        samples_resized = ra.resize_progressive_output(samples)

        if ra.condition is not None:
            conds_ = ra.condition(conds)
        else:
            conds_ = conds

        mean_logvar, *_ = ra.encoder([samples_resized, conds_], training=False)
        z = ra.reparameterize(mean_logvar, training=False)
        reconstructed = ra.decoder([z, conds_], training=False)

        error = ra.reconstruction_loss(samples_resized, reconstructed)

    samples_resized = samples_resized.cpu().numpy()
    reconstructed = reconstructed.cpu().numpy()
    error = error.cpu().numpy()
    z_np = z.cpu().numpy()

    height, width = samples_resized.shape[1:3]
    ar = width / height

    figwidth = 4
    figheight = figwidth / ar

    n_samples = samples.shape[0]

    # 4 columns: original, reconstructed, difference, latent histogram
    fig, axs = plt.subplots(
        n_samples,
        4,
        figsize=(4 * figwidth, n_samples * figheight),
    )

    # Handle single row case
    if n_samples == 1:
        axs = axs.reshape(1, -1)

    for row, (real, rec, err, z_sample) in enumerate(
        zip(samples_resized, reconstructed, error, z_np)
    ):

        if row >= n_samples:
            break

        if row == 0:
            axs[row, 0].set_title("Original", fontsize=12, pad=15)
            axs[row, 1].set_title("Reconstruction", fontsize=12, pad=15)
            axs[row, 2].set_title("Anomaly Map", fontsize=12, pad=15)
            axs[row, 3].set_title("Latent Distribution", fontsize=12, pad=15)

        # First column: keep axis on for ylabel, hide ticks, rotate 90 degrees
        axs[row, 0].set_ylabel(
            f"{row+1}",
            fontsize=10,
            rotation=90,
            labelpad=15,
            ha="center",
            va="center",
        )

        # Image columns: hide ticks
        for col in range(3):
            axs[row, col].set_xticks([])
            axs[row, col].set_yticks([])

        axs[row, 0].imshow(real, cmap="gray")
        axs[row, 1].imshow(rec, cmap="gray")
        axs[row, 2].imshow(err, cmap="inferno")

        # Plot histogram of latent variables
        axs[row, 3].hist(z_sample.flatten(), bins=50, alpha=0.7, density=True)
        axs[row, 3].set_xlabel("z value")
        axs[row, 3].set_ylabel("Density")
        axs[row, 3].set_xlim(left=-4, right=4)
        axs[row, 3].set_ylim(bottom=0, top=1.0)

        # Overlay standard normal for comparison
        axs[row, 3].plot(x, standard_normal, "r--", alpha=0.8, label="N(0,1)")
        axs[row, 3].legend()

    epoch_num = step + 1
    filename = f"epoch_{epoch_num:05d}.jpg"
    title = f"Reconstruction Results - Epoch {epoch_num}"

    # Add overall figure title
    fig.suptitle(title, fontsize=14, y=0.96)

    plt.tight_layout()
    plt.subplots_adjust(top=0.92, left=0.08)

    fig.savefig(rec_dir / filename)

    plt.close(fig)

    print(f"Epoch {step + 1}: reconstruction results saved")

In [ ]:
from keras.callbacks import ModelCheckpoint, LambdaCallback

from agx_torch.callbacks import (
    GarbageCollectionCallback,
    MemorySnapshotCallback,
    MemoryLeakDiagnosticCallback,
)
from agx_core.models.reversed_autoencoder.callbacks import (
    AdversarialEquilibriumCallback,
    ProgressiveGrowingCallback,
    BackboneThawCallback,
)

callbacks = [
    LambdaCallback(on_epoch_end=plot_on_step_end),
    AdversarialEquilibriumCallback(
        0.5, -0.5, min_pause_steps=165, warmup_pause_steps=660
    ),
    # ProgressiveGrowingCallback(3000, 3000),
    BackboneThawCallback(patience=660),
    ModelCheckpoint(
        filepath="ra_mbnetv3.best.keras",
        monitor="val_loss_rec",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    # MemorySnapshotCallback((130, 260, 650), 26),
    # MemoryLeakDiagnosticCallback(39, 13, 130, True),
    GarbageCollectionCallback(330, 495, True),
]

In [ ]:
history = ra.fit(
    loader_train,
    validation_data=loader_valid,
    initial_epoch=0,
    epochs=100,
    callbacks=callbacks,
)

In [ ]:
ra.save("ra_mbnetv3.model.keras")

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(history.history)

hist_file = Path("history.csv")
if hist_file.exists():
    hist = pd.read_csv(hist_file)
    df.index += len(hist)
    hist = pd.concat([hist, df])
    # hist.to_csv(hist_file, index=False)
else:
    df.to_csv(hist_file, index=False)
    hist = df
hist.tail()

In [ ]:
loader_test = DataLoader(ds_test, batch_size=16)

In [ ]:
import cv2
from keras import ops, models
import matplotlib.pyplot as plt


from agx_core.models.reversed_autoencoder.model import kl_divergence
from agx_core.models.reversed_autoencoder.anomaly import anomaly_pipeline
from agx_core.models.reversed_autoencoder.anomaly_viz import AnomalyExplorer

ra_best = ra  # models.load_model("ra_mbnetv3.best.keras")
explorer = AnomalyExplorer(
    model=ra_best,
    dataloader=loader_test,
    n_samples=9,
    device="cpu",
    # use_feature_maps=True,
)
explorer.show()

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(21, 14))

# Adjust val_loss_embed, forgot to update validation scaling
hist[["elbo_real", "kld_real"]].plot(ax=ax)

In [ ]:
from agx_core.models.reversed_autoencoder.model import kl_divergence

with torch.device("cpu"):
    ra.eval()
    I = torch.rand((1, *img_shape[1:]))
    C = torch.rand((1, *cond_shape[1:]))

    import keras

    i_ = keras.Input(img_shape[1:])
    c_ = keras.Input(cond_shape[1:])

    c1_ = ra.condition(c_) if ra.condition else c_

    rec, mu_logvar, *_ = ra([i_, c_])
    kld = kl_divergence(*mu_logvar)

    model = keras.Model(
        inputs=[i_, c_], outputs=[rec, kld], name="reversed_autoencoder"
    )
    model.eval()

    torch.onnx.export(
        model,
        ([I, C],),
        "model.onnx",
        input_names=["image", "condition"],
        output_names=["reconstruction", "kld"],
    )

In [ ]:
import matplotlib.pyplot as plt

from agx_core.helpers import _channel_axis
from keras import ops


def kl_divergence(mean, logvar):
    return 0.5 * ops.sum(
        ops.square(mean) + ops.exp(logvar) - logvar - 1.0,
        axis=_channel_axis(),
    )


fig, axes = plt.subplots(5, 3, figsize=(12, 20))

for idx in range(5):
    (I, C), y = ds_test[idx]
    I = I.cpu().detach().numpy()
    rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]]).cpu().detach().numpy()[0]
    D = mse_weighted(I[np.newaxis, ...], rec[np.newaxis, ...], 7).cpu().detach().numpy()[0]
    
    (mean, logvar), _ = ra.encoder([I[np.newaxis, ...], C[np.newaxis, ...]])
    kld = kl_divergence(mean, logvar).cpu().detach().numpy()[0]
    
    axes.flat[3 * idx].imshow(rec, cmap="gray")
    axes.flat[3 * idx + 1].imshow(I, cmap="gray")
    axes.flat[3 * idx + 2].imshow(kld, cmap="inferno")
    axes.flat[3 * idx].axis("off")
    axes.flat[3 * idx + 1].axis("off")
    axes.flat[3 * idx + 2].axis("off")

plt.tight_layout()
plt.show()